# Pb4U-GNet Inference Pipeline

This notebook provides the steps to run inference using Pb4U-GNet.

Make sure you set these two environmental variables:
* `PB4U_PROJECT` should point to the repository root.
* `PB4U_DATA` should point to your data directory.

In [ ]:
import os
from pathlib import Path

os.environ["PB4U_PROJECT"] = "/path/to/pb4u"
os.environ["PB4U_DATA"] = "/path/to/pb4u/data"

PB4U_PROJECT = os.environ["PB4U_PROJECT"]
PB4U_DATA = os.environ["PB4U_DATA"]

## 1. Prepare Pose Sequences
Convert a sequence from the VTO or AMASS dataset into a `.pkl` file formatted for the network.

In [ ]:
from utils.data_making import convert_vto_to_pkl, convert_amass_to_pkl

# Example for VTO dataset conversion
VTO_DATASET_PATH = '/path/to/vto'
vto_sequence_path = Path(VTO_DATASET_PATH) / 'tshirt/simulations/tshirt_shape00_07_02.pkl'
target_pkl_path =  Path(PB4U_DATA) / 'temp/07_02.pkl'

convert_vto_to_pkl(vto_sequence_path, target_pkl_path, n_zeropose_interpolation_steps=30)
print(f'Pose sequence saved into {target_pkl_path}')

Pose sequence saved into /home/adam/research/pb4u/data/temp/07_02.pkl


## 2. Generate Pb4U-GNet Rollout
Load the resolution-adaptive model

In [ ]:
import torch
from utils.validation import Config as ValidationConfig
from utils.validation import load_runner_from_checkpoint, update_config_for_validation, create_one_sequence_dataloader
from utils.arguments import load_params
from utils.common import move2device, pickle_dump

# Material parameters
config_dict = {
    'density': 0.20022,
    'lame_mu': 23600.0,
    'lame_lambda': 44400,
    'bending_coeff': 3.962e-05,
    'separate_arms': True,
    'garment_dict_file': 'garments_dict.pkl',
    'smpl_model': 'smpl/SMPL_FEMALE.pkl'
}
validation_config = ValidationConfig(**config_dict)

# Load Pb4U-GNet Config and Checkpoint
config_name = 'pb4u'  # Update with your specific Pb4U config name
checkpoint_path = Path(PB4U_DATA) / 'trained_models' / 'pb4u.pth'

modules, experiment_config = load_params(config_name)
experiment_config = update_config_for_validation(experiment_config, validation_config)
runner_module, runner = load_runner_from_checkpoint(checkpoint_path, modules, experiment_config)

Load the pose sequence for validation

In [5]:
sequence_path =  Path(PB4U_DATA) / 'temp/07_02.pkl'
garment_name = 'tshirt'

dataloader = create_one_sequence_dataloader(sequence_path, garment_name, modules, experiment_config)
sequence = next(iter(dataloader))
sequence = move2device(sequence, 'cuda:0')

trajectories_dict = runner.valid_rollout(sequence, bare=False)

100%|███████████████████████████████████████████| 112/112 [00:30<00:00,  3.63it/s]


Save the output sequence to disc

In [6]:
out_path = Path(PB4U_PROJECT) / 'rendering' / 'output.pkl'
pickle_dump(dict(trajectories_dict), out_path)
print(f"Rollout saved into {out_path}")

Rollout saved into /home/adam/research/pb4u/rendering/output.pkl


## 4. Metrics & Evaluation

The rollout stores per-frame values for each of the six physics-based loss terms used during training. The cell below reports their temporal mean.

| Key | Measures |
|---|---|
| `stretching_energy_loss` | St. Venant–Kirchhoff in-plane stretch / compression |
| `bending_energy_loss` | Curvature penalty between adjacent faces |
| `collision_penalty_loss` | Garment–body interpenetration depth |
| `gravitational_energy_loss` | Penalty for vertices raised above rest height |
| `friction_energy_loss` | Tangential sliding at garment–body contact |
| `inertia_loss` | Temporal coherence via velocity-change magnitude |

In [5]:
METRIC_LABELS = {
    'stretching_energy_loss':    'Stretching',
    'bending_energy_loss':       'Bending',
    'collision_penalty_loss':    'Collision',
    'gravitational_energy_loss': 'Gravity',
    'friction_energy_loss':      'Friction',
    'inertia_loss':              'Inertia',
}

col_w = 14
print(f"  {'Metric':<22}  {'Mean':>{col_w}}")
print("  " + "-" * (22 + col_w + 4))
for key, values in trajectories_dict['metrics'].items():
    mean = sum(values) / len(values)
    label = METRIC_LABELS.get(key, key)
    print(f"  {label:<22}  {mean:>{col_w}.6e}")

  Metric                            Mean
  ----------------------------------------
  Stretching                2.433832e-02
  Inertia                   1.301507e-03
  Gravity                   8.824197e-02
  Collision                 4.072859e-04
  Bending                   3.036286e-03
  Friction                  1.237884e-03


## 5. Render Video

Rendering is a two-step pipeline run from the terminal.

**Step 1 — Export per-frame OBJ files** from the saved rollout:

```sh
python rendering/output.py
```

This reads `output.pkl` from the current directory and writes per-frame `garment_XXXX.obj` and `body_XXXX.obj` files into `obj_frames/`.

**Step 2 — Render with Blender** (must be installed and on `PATH`):

```sh
blender --background rendering/scene.blend --python rendering/render.py -- --path obj_frames/
```

Rendered frames are saved to `obj_frames/render/` and assembled into `output.mp4`.